In [1]:
# ==============================================================================
# Setup & Dependencies
# ==============================================================================
%load_ext autoreload
%autoreload 2

import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController, RandomController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_4
from src.rl.trainer_team import train_team_ppo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

⚡ Device: cuda


## Phase A: Train against random bot

In [10]:
# ==============================================================================
# Configuration & Directory Setup
# ==============================================================================
STAGE = 4
TEAM_SIZE = 2      # 2v2 format
NUM_ENVS = 16
MAX_STEPS = 1800   # 30.0s at 60 Hz
TIME_LIMIT = 30.0

OBS_DIM = 80       # Local actor observation
STATE_DIM = 32     # Centralized critic global state

SAVE_DIR = f"models/stage{STAGE}_phaseA"
POOL_DIR = f"models/stage{STAGE}_phaseA/pool"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(POOL_DIR, exist_ok=True)

In [8]:
# ==============================================================================
# Environment Factory
# ==============================================================================
def make_env(env_idx: int):
    def _init():
        import torch
        torch.set_num_threads(1)

        # 50/50 balance between Red and Blue teams across environments
        is_red = env_idx % 2 == 0
        learner_team = "red" if is_red else "blue"
        opp_team = "blue" if is_red else "red"

        # Anchored Baseline: Train strictly against 100% Heuristic bots
        #opp_ctrl = HeuristicBotController(TeamHeuristicCoordinator(team=opp_team))
        opp_ctrl = RandomController()

        roster = []
        for i in range(TEAM_SIZE):
            roster.append(
                PlayerSlot(
                    learner_team,
                    PlayerStats(name=f"Learner_{i}", accel=3200.0),
                    controller="RL",
                )
            )
        for i in range(TEAM_SIZE):
            roster.append(
                PlayerSlot(
                    opp_team,
                    PlayerStats(name=f"Opponent_{i}", accel=3200.0),
                    controller=opp_ctrl,
                )
            )

        cfg = MatchConfig(
            mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99),
            roster=roster,
        )

        return MatchEnv(
            match_config=cfg,
            reward_shaper=DenseReward_4(team=learner_team),
            reset_strategy=RandomReset(),
            learner_team=learner_team,
            max_steps=MAX_STEPS,
        )

    return _init

# Parallelize 16 environments across CPU cores
train_envs = gym.vector.AsyncVectorEnv(
    [make_env(i) for i in range(NUM_ENVS)], 
    context="spawn"
)

In [9]:
# ==============================================================================
# Model Initialization & Selective Weight Loading
# ==============================================================================
# Instantiate dual-tower MAPPO model: 80d Actor + 32d Critic
model = ActorCritic(obs_dim=OBS_DIM, state_dim=STATE_DIM).to(device)

stage3_best = "models/stage3/best_model.pt"
if os.path.exists(stage3_best):
    # Bootstrap Actor weights from Stage 3 while keeping the 32d Critic fresh
    model.load_actor_weights(stage3_best, device=device)
else:
    print(f"⚠️ Warning: '{stage3_best}' not found. Training from scratch.")

✅ Bootstrapped Decentralized Actor weights from: models/stage3/best_model.pt


In [10]:
# ==============================================================================
# MAPPO Training Loop
# ==============================================================================
train_team_ppo(
    envs=train_envs,
    model=model,
    baseline_type="random",
    device=device,
    team_size=TEAM_SIZE,
    double_eval=False,          # Rank models purely against the Heuristic baseline
    total_timesteps=15_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
    lr_initial=1.5e-5,            # Higher initial LR to overcome 1v1 local minima
    lr_final=5e-6,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef_initial=0.006,
    ent_coef_final=0.002,
)

train_envs.close()

🚀 MAPPO Training | Format: 2v2 | Envs: 16 | Batch: 8192

📊 [EVALUATION @ Step  106496 | Target: random 2v2]
   ⚔️  Record : 43/50 Wins ( 86.0%) | Scored In: 44/50 matches
   ⚽ Goals  : 192 Scored, 4 Conceded (+188 net) | Reward: 800.462
   📦 POOL UPDATED: Cached new generational champion -> models/stage4_phaseA/pool/history_106496.pt
   ⭐⭐ PROMOTED! New Best Score -> Saved: models/stage4_phaseA/best_model.pt
      [WR: 86.0% (was N/A) | Scored: 44 (was N/A) | Net: +188 (was N/A) | Rew: 800.462 (was N/A)]

📊 [EVALUATION @ Step  204800 | Target: random 2v2]
   ⚔️  Record : 42/50 Wins ( 84.0%) | Scored In: 42/50 matches
   ⚽ Goals  : 178 Scored, 0 Conceded (+178 net) | Reward: 779.824
   ❌ RETAINING CURRENT BEST. Current score did not beat: [WR: 86.0%, Scored: 44, Net: +188, Rew: 800.462]

📊 [EVALUATION @ Step  303104 | Target: random 2v2]
   ⚔️  Record : 36/50 Wins ( 72.0%) | Scored In: 36/50 matches
   ⚽ Goals  : 119 Scored, 4 Conceded (+115 net) | Reward: 520.721
   ❌ RETAINING CURRENT

In [5]:
# ==============================================================================
# Replay Generation
# ==============================================================================
from src.rl.evaluator import evaluate_and_generate_html_2

replay_path = evaluate_and_generate_html_2(
    red_agent=f"{SAVE_DIR}/best_model.pt",
    blue_agent="heuristic",
    team_size=TEAM_SIZE,
    num_episodes=5,
    output_dir="render/",
    filename="stage4_mappo_vs_heuristic.html",
    device=device,
)

🎬 Multi-Agent Replay generated: /home/minh-quan/Documents/Haxball project/training/render/stage4_mappo_vs_heuristic.html


## Phase B: Train against heuristic bot

In [2]:
# ==============================================================================
# Configuration & Directory Setup
# ==============================================================================
STAGE = 4
TEAM_SIZE = 2      # 2v2 format
NUM_ENVS = 16
MAX_STEPS = 1800   # 30.0s at 60 Hz
TIME_LIMIT = 30.0

OBS_DIM = 80       # Local actor observation
STATE_DIM = 32     # Centralized critic global state

SAVE_DIR = f"models/stage{STAGE}_phaseB"
POOL_DIR = f"models/stage{STAGE}_phaseB/pool"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(POOL_DIR, exist_ok=True)

In [3]:
# ==============================================================================
# Environment Factory
# ==============================================================================
def make_env(env_idx: int):
    def _init():
        import torch
        torch.set_num_threads(1)

        # 50/50 balance between Red and Blue teams across environments
        is_red = env_idx % 2 == 0
        learner_team = "red" if is_red else "blue"
        opp_team = "blue" if is_red else "red"

        # Anchored Baseline: Train strictly against 100% Heuristic bots
        #opp_ctrl = HeuristicBotController(TeamHeuristicCoordinator(team=opp_team))
        #opp_ctrl = RandomController()
        opp_ctrl = PoolController(
            pool_dir=POOL_DIR,
            team=opp_team,
            #device="cpu",
            heuristic_pct=0.4,
        )

        roster = []
        for i in range(TEAM_SIZE):
            roster.append(
                PlayerSlot(
                    learner_team,
                    PlayerStats(name=f"Learner_{i}", accel=3200.0),
                    controller="RL",
                )
            )
        for i in range(TEAM_SIZE):
            roster.append(
                PlayerSlot(
                    opp_team,
                    PlayerStats(name=f"Opponent_{i}", accel=3200.0),
                    controller=opp_ctrl,
                )
            )

        cfg = MatchConfig(
            mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99),
            roster=roster,
        )

        return MatchEnv(
            match_config=cfg,
            reward_shaper=DenseReward_4(team=learner_team),
            reset_strategy=RandomReset(),
            learner_team=learner_team,
            max_steps=MAX_STEPS,
        )

    return _init

# Parallelize 16 environments across CPU cores
train_envs = gym.vector.AsyncVectorEnv(
    [make_env(i) for i in range(NUM_ENVS)], 
    context="spawn"
)

In [4]:
# ==============================================================================
# Model Initialization & Selective Weight Loading
# ==============================================================================
# Instantiate dual-tower MAPPO model: 80d Actor + 32d Critic
model = ActorCritic(obs_dim=OBS_DIM, state_dim=STATE_DIM).to(device)
stage3_best = "models/stage3/best_model.pt"
if os.path.exists(stage3_best):
    # Bootstrap Actor weights from Stage 3 while keeping the 32d Critic fresh
    model.load_actor_weights(stage3_best, device=device)
else:
    print(f"⚠️ Warning: '{stage3_best}' not found. Training from scratch.")

✅ Bootstrapped Decentralized Actor weights from: models/stage3/best_model.pt


In [ ]:
# ==============================================================================
# MAPPO Training Loop
# ==============================================================================
train_team_ppo(
    envs=train_envs,
    model=model,
    device=device,
    team_size=TEAM_SIZE,
    baseline_type="heuristic",    
    pretrained_model_path="models/stage3/best_model.pt",
    warmup_steps=1_500_000,       # Freezes Actor for first 1500k steps
    lr_actor_initial=1.5e-5,    # Gentle fine-tuning for Stage 3 reflexes
    ppo_epochs_post=2,
    lr_actor_final=3e-6,
    lr_critic_initial=3e-4,     # Fast learning rate for the fresh 32d Critic
    lr_critic_final=1e-5,
    total_timesteps=25_000_000,
    kl_coef=0.01,
    eval_freq=100_000,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
)

train_envs.close()

⚓ KL Anchor Active: Policy regularized against Stage 3 champion: models/stage3/best_model.pt
🚀 MAPPO Training | Format: 2v2 | Envs: 16 | Batch: 8192
⚡ Two-Speed Optimizer: Actor LR = 1.5e-05 -> 3.0e-06 | Critic LR = 3.0e-04 -> 1.0e-05
🛡️  Warm-up: 1,500,000 steps | KL Penalty Weight: 0.05 | Post-Warmup Epochs: 1

📊 [EVALUATION @ Step  106496 [WARM-UP] | Target: heuristic 2v2]
   ⚔️  Record : 34/50 Wins ( 68.0%) | Scored In: 36/50 matches
   ⚽ Goals  : 54 Scored, 10 Conceded (+44 net) | Reward: 115.730
   📦 POOL UPDATED: Cached new generational champion -> models/stage4_phaseB/pool/history_106496.pt
   ⭐⭐ PROMOTED! New Best Score -> Saved: models/stage4_phaseB/best_model.pt
      [WR:  68.0% (was N/A) | Scored: 36 (was N/A) | Net: +44 (was N/A) | Rew: 115.730 (was N/A)]

📊 [EVALUATION @ Step  204800 [WARM-UP] | Target: heuristic 2v2]
   ⚔️  Record : 34/50 Wins ( 68.0%) | Scored In: 36/50 matches
   ⚽ Goals  : 54 Scored, 10 Conceded (+44 net) | Reward: 115.730
   ❌ RETAINING CURRENT BEST

KeyboardInterrupt: 